# Your data from interaction to rows
**Week 6 · 5:30–5:40 PM · Four code cells**

Run the cells from top to bottom. Use a prediction CSV exported from your EZ-MMLA tool. This notebook does not train a model or calculate participation.

**Before upload:** Colab processes the file in a cloud runtime. Upload only if everyone whose data are included agrees. Otherwise open the CSV locally or inspect the live output and discuss the questions below. Avoid uploading raw video or names.

[EZ-MMLA tools](https://mmla.gse.harvard.edu/tools/) · [Run RTMPose](https://mmla.gse.harvard.edu/tools/rtmpose/run/) · [RTMPose export documentation](https://mmla.gse.harvard.edu/tools/rtmpose/)

For RTMPose: **Download → Download Predictions**. Other exports may have different schemas. No example data are silently substituted if a file fails to load.

In [ ]:
# 1. Upload one prediction CSV (Colab).
import io
import pandas as pd
from IPython.display import display
from google.colab import files

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one CSV, then rerun this cell.")
filename, raw = next(iter(uploaded.items()))
if not filename.lower().endswith(".csv"):
    raise ValueError("Choose the prediction .csv file, not a video or notebook.")
try:
    df = pd.read_csv(io.BytesIO(raw), sep=None, engine="python", encoding="utf-8-sig")
except (pd.errors.EmptyDataError, pd.errors.ParserError, UnicodeDecodeError, ValueError) as exc:
    raise ValueError("Could not read this export. Inspect it locally or use the live output.") from exc
df.columns = [str(c).strip() for c in df.columns]
if df.empty:
    raise ValueError("The CSV has no data rows. Collect predictions again or use the live output.")
print(f"Loaded {filename}: {len(df):,} rows and {len(df.columns)} columns")

In [ ]:
# 2. What does one row represent? Look before interpreting.
display(df.head())
print("One row represents: _________________________________")

In [ ]:
# 3. Dimensions, columns, and missing values.
print("Shape (rows, columns):", df.shape)
structure = pd.DataFrame({
    "type": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_percent": (100 * df.isna().mean()).round(1)
})
with pd.option_context("display.max_rows", None):
    display(structure)
print("Missing here means an empty/NA export cell, not automatically no behavior.")

In [ ]:
# 4. Optional score thresholds and exported frame structure.
# Candidate names are a convenience; inspect the tool documentation before interpretation.
import numpy as np
score_cols = [c for c in df.columns
              if c.lower().endswith(("_score", "_confidence"))
              or c.lower() in ("score", "confidence")]
if score_cols:
    scores = df[score_cols].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    observed = scores.stack().dropna()
    print("Candidate score columns:", score_cols)
    print(f"Available numeric scores: {observed.size:,} / {scores.size:,} exported score cells")
    if observed.empty:
        print("No numeric scores to compare.")
    elif not observed.between(0, 1).all():
        print("Scores are not all in [0, 1]. Check their meaning; thresholds skipped.")
    else:
        threshold_summary = pd.DataFrame([
            {"threshold": t,
             "available_scores_at_or_above": int((observed >= t).sum()),
             "percent_of_available_scores": round(100 * (observed >= t).mean(), 1)}
            for t in [0.50, 0.80]
        ])
        display(threshold_summary)
        print("Denominator: available numeric scores, not people or participation.")
        print("Scores removed before export cannot be recovered by lowering this threshold.")
else:
    print("No score/confidence columns recognized. Inspect your tool's actual output instead.")

cols = {c.lower(): c for c in df.columns}
frame_col = next((cols[k] for k in ["frame", "frame_id", "frame_index"] if k in cols), None)
person_col = next((cols[k] for k in ["person", "person_id"] if k in cols), None)
if frame_col:
    grouped = df.groupby(frame_col, dropna=True)
    counts = grouped.size().rename("exported_rows")
    if person_col:
        counts = pd.concat([counts, grouped[person_col].nunique().rename("distinct_person_values")], axis=1)
    display(counts.head(15))
    print("Only exported frames appear here. Missing frames are not filled with zeros.")
    print("Person values alone do not establish stable student identity across frames.")
else:
    print("No frame column recognized. Use the first rows to discuss the unit of observation.")

## Return to the worksheet
1. What had to happen between the interaction and one row?
2. Which decisions were already made before export?
3. What claim does one concrete output support? What additional evidence would a stronger claim need?

**No CSV?** Inspect the live output or a locally opened file: identify one output unit, something it captures, and something it leaves unresolved. Do not infer model results you did not observe.

**RTMPose note:** Its documentation describes one detected person per frame per row and blank keypoint cells below the export threshold. A person index does not by itself establish identity tracking. These meanings should not be assumed for every other tool.

The Teachable Machine test uses newly supplied examples. It is not a synchronized second measurement of the original three-minute interaction.

After class, follow the group's agreed retention plan for downloaded files, Colab copies, and any saved outputs.